# Unboxing the Transformer: Decoding Strategies from Scratch

This notebook looks under the hood of a decoder-only LLM (GPT-2), bypassing the
high-level `.generate()` API to implement the core sampling mechanics manually:

- **Tokenization** — how raw text becomes integer IDs
- **Logits** — the raw, unnormalized next-token scores a model produces
- **Greedy decoding** — always picking the highest-scoring token
- **Temperature scaling** — controlling randomness in the output distribution
- **Top-K sampling** — truncating to the K most likely tokens
- **Top-P (nucleus) sampling** — truncating to the smallest set of tokens covering probability mass P
- **Autoregressive generation** — combining all of the above into a full generation loop

## 1. Setup & Imports

In [1]:
import random

import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

## 2. Configuration & Reproducibility

We fix all relevant random seeds and resolve the compute device up front so
that sampling-based generation (Section 6 onward) is reproducible run-to-run.

In [2]:
SEED = 42
MODEL_NAME = "gpt2"


def set_seed(seed: int) -> None:
    """Seed all relevant RNGs for reproducible sampling.

    Args:
        seed: The seed value to apply across Python, NumPy, and PyTorch RNGs.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


## 3. Model & Tokenizer Loading

We load GPT-2 and its tokenizer, then switch the model to `eval()` mode. This
disables dropout and other training-only behaviors so that repeated forward
passes on the same input are deterministic.

In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
model.eval()

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

## 4. Data Processing: The Tokenizer Grid

Models don't read text; they read arrays of integers. Before looking at any
model output, we inspect exactly how our prompt is tokenized and mapped to IDs.

In [4]:
prompt = "What is the capital of France? It is the city of"
inputs = tokenizer(prompt, return_tensors="pt").to(device)

print("Original Text:", prompt)
print("Input IDs Shape:", inputs["input_ids"].shape)
print("Input IDs Tensor:", inputs["input_ids"])

# Decode each token individually to make the text <-> ID mapping explicit.
print("\nToken Mapping:")
for token_id in inputs["input_ids"][0]:
    print(f"{token_id.item()} -> '{tokenizer.decode(token_id)}'")

Original Text: What is the capital of France? It is the city of
Input IDs Shape: torch.Size([1, 12])
Input IDs Tensor: tensor([[2061,  318,  262, 3139,  286, 4881,   30,  632,  318,  262, 1748,  286]])

Token Mapping:
2061 -> 'What'
318 -> ' is'
262 -> ' the'
3139 -> ' capital'
286 -> ' of'
4881 -> ' France'
30 -> '?'
632 -> ' It'
318 -> ' is'
262 -> ' the'
1748 -> ' city'
286 -> ' of'


## 5. Raw Logits & Greedy Decoding

Passing the input through the model doesn't produce a word — it produces a
raw, unnormalized score (a *logit*) for every token in the vocabulary. The
simplest way to pick a token is **greedy decoding**: take the index with the
single highest logit.

In [5]:
with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits
print(f"Logits Tensor Shape: {logits.shape}")  # [batch_size, seq_len, vocab_size]

# We only care about predicting the NEXT token, so we isolate the logits
# for the final position in the sequence.
next_token_logits = logits[0, -1, :]
print(f"Next Token Logits Shape: {next_token_logits.shape}")

best_token_id = torch.argmax(next_token_logits).item()
best_score = next_token_logits[best_token_id].item()
predicted_word = tokenizer.decode(best_token_id)

print(f"\nHighest Logit Score: {best_score:.4f}")
print(f"Predicted Token ID: {best_token_id}")
print(f"Predicted Word: '{predicted_word}'")
print(f"\nResult: {prompt}{predicted_word}")

Logits Tensor Shape: torch.Size([1, 12, 50257])
Next Token Logits Shape: torch.Size([50257])

Highest Logit Score: -77.8067
Predicted Token ID: 6342
Predicted Word: ' Paris'

Result: What is the capital of France? It is the city of Paris


## 6. Decoding Strategies

Greedy decoding is deterministic and often repetitive. The functions below
implement the three standard controls used to introduce and shape randomness
in generation: temperature scaling, Top-K masking, and Top-P (nucleus) masking.

A small synthetic logits tensor (vocab size 10) is used to unit-test each
function before applying them to the live GPT-2 model.

In [6]:
# Synthetic logits over a toy 10-word vocabulary, used to validate each
# decoding function in isolation before wiring them into the full model.
test_logits = torch.tensor([2.0, 0.5, -1.0, 8.0, 5.0, 4.5, -3.0, 1.0, 0.0, 7.5])

### 6.1 Temperature Scaling

$$ p_i = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)} $$

A high temperature ($T > 1$) flattens the distribution (more random output);
a low temperature ($T < 1$) sharpens it toward greedy behavior. As $T \to 0$,
the distribution collapses onto the argmax, so we special-case very small
temperatures to avoid a division that would blow up numerically.

In [7]:
def apply_temperature(logits: torch.Tensor, temperature: float = 1.0) -> torch.Tensor:
    """Convert raw logits into a probability distribution scaled by temperature.

    Args:
        logits: 1D tensor of raw, unnormalized scores.
        temperature: Controls randomness. Values < 1 sharpen the distribution
            toward greedy decoding; values > 1 flatten it toward uniform.

    Returns:
        A 1D tensor of probabilities summing to 1.
    """
    # Near-zero temperature is equivalent to greedy decoding. Dividing by a
    # value this small would otherwise overflow, so we short-circuit to a
    # one-hot distribution on the argmax.
    if temperature < 1e-4:
        probs = torch.zeros_like(logits)
        probs[torch.argmax(logits)] = 1.0
        return probs

    scaled_logits = logits / temperature
    probs = F.softmax(scaled_logits, dim=-1)
    return probs


print("T=1.0 (Standard):", apply_temperature(test_logits, 1.0))
print("T=0.1 (Greedy):", apply_temperature(test_logits, 0.1))
print("T=5.0 (Random):", apply_temperature(test_logits, 5.0))

T=1.0 (Standard): tensor([1.4659e-03, 3.2709e-04, 7.2983e-05, 5.9139e-01, 2.9444e-02, 1.7858e-02,
        9.8772e-06, 5.3928e-04, 1.9839e-04, 3.5870e-01])
T=0.1 (Greedy): tensor([8.6979e-27, 2.6607e-33, 8.1392e-40, 9.9331e-01, 9.2950e-14, 6.2629e-16,
        0.0000e+00, 3.9488e-31, 1.7928e-35, 6.6929e-03])
T=5.0 (Random): tensor([0.0717, 0.0531, 0.0394, 0.2381, 0.1307, 0.1183, 0.0264, 0.0587, 0.0481,
        0.2155])


### 6.2 Top-K Masking

Temperature alone still leaves every token with some nonzero probability of
being sampled. Top-K truncates the tail: only the K highest-scoring tokens
remain eligible, and everything else is masked to `-inf` so it contributes
zero probability after softmax.

In [8]:
def apply_top_k(logits: torch.Tensor, k: int = 5) -> torch.Tensor:
    """Mask all but the top-k logits with -inf.

    Args:
        logits: 1D tensor of raw scores.
        k: Number of highest-scoring tokens to keep eligible.

    Returns:
        A 1D tensor the same shape as `logits`, with all but the top-k
        values replaced by -inf.
    """
    if k <= 0:
        return logits

    # The k-th largest value acts as the cutoff threshold; anything below
    # it gets masked out.
    top_k_values, _ = torch.topk(logits, k)
    threshold = top_k_values[-1]

    mask = logits < threshold
    logits = logits.masked_fill(mask, -float("inf"))
    return logits


print("Original:", test_logits)
print("Top-K (k=3):", apply_top_k(test_logits.clone(), k=3))

Original: tensor([ 2.0000,  0.5000, -1.0000,  8.0000,  5.0000,  4.5000, -3.0000,  1.0000,
         0.0000,  7.5000])
Top-K (k=3): tensor([  -inf,   -inf,   -inf, 8.0000, 5.0000,   -inf,   -inf,   -inf,   -inf,
        7.5000])


### 6.3 Top-P (Nucleus) Masking

Top-K is rigid — a fixed K may be too large when the model is confident, or
too small when it's uncertain. Top-P dynamically selects the smallest set of
tokens whose cumulative probability exceeds threshold $P$, so the eligible
set shrinks and grows with the model's confidence.

In [9]:
def apply_top_p(logits: torch.Tensor, p: float = 0.9) -> torch.Tensor:
    """Mask logits outside the smallest set covering cumulative probability p.

    Args:
        logits: 1D tensor of raw scores.
        p: Cumulative probability threshold (nucleus size), in (0, 1].

    Returns:
        A 1D tensor the same shape as `logits`, with tokens outside the
        nucleus replaced by -inf.
    """
    # Sort descending so cumulative probability accumulates from the most
    # likely token first; keep original indices to scatter back later.
    sorted_logits, sorted_indices = torch.sort(logits, descending=True)
    sorted_probs = F.softmax(sorted_logits, dim=-1)
    cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

    # Tokens where the *cumulative* probability already exceeds p are
    # outside the nucleus and should be dropped.
    sorted_mask = cumulative_probs > p

    # Shift the mask right by one position so we always keep the token that
    # first crosses the threshold -- without this, a nucleus could end up
    # empty when the very first token already exceeds p.
    sorted_mask[..., 1:] = sorted_mask[..., :-1].clone()
    sorted_mask[..., 0] = False

    sorted_logits = sorted_logits.masked_fill(sorted_mask, -float("inf"))

    # Scatter back into the original (unsorted) vocabulary order.
    logits = torch.zeros_like(logits).scatter(-1, sorted_indices, sorted_logits)
    return logits


print("Top-P (p=0.9):", apply_top_p(test_logits.clone(), p=0.9))

Top-P (p=0.9): tensor([  -inf,   -inf,   -inf, 8.0000,   -inf,   -inf,   -inf,   -inf,   -inf,
        7.5000])


## 7. Autoregressive Generation Loop

With all three controls implemented, we assemble them into a full
autoregressive loop: at each step, mask the logits (Top-K, then Top-P),
convert the survivors to a temperature-scaled distribution, sample one
token, and append it to the running sequence.

In [10]:
def generate_text(
    prompt: str,
    max_tokens: int = 30,
    temperature: float = 0.7,
    top_k: int = 50,
    top_p: float = 0.9,
) -> str:
    """Autoregressively generate text from a prompt using GPT-2.

    Args:
        prompt: The seed text to continue.
        max_tokens: Number of new tokens to generate.
        temperature: Softmax temperature applied after masking.
        top_k: Top-K cutoff; set to 0 to disable.
        top_p: Top-P (nucleus) cutoff; set to 1.0 to disable.

    Returns:
        The prompt concatenated with the generated continuation.
    """
    input_ids = tokenizer(prompt, return_tensors="pt")["input_ids"].to(device)

    for _ in range(max_tokens):
        with torch.no_grad():
            logits = model(input_ids).logits[0, -1, :]

        # 1. Masking: narrow the candidate pool before sampling.
        if top_k > 0:
            logits = apply_top_k(logits, k=top_k)
        if top_p < 1.0:
            logits = apply_top_p(logits, p=top_p)

        # 2. Convert surviving logits into a proper probability distribution.
        probs = apply_temperature(logits, temperature=temperature)

        # 3. Sample one token from that distribution (not argmax -- this is
        #    what makes generation stochastic rather than greedy).
        next_token = torch.multinomial(probs, num_samples=1).unsqueeze(0)

        # 4. Append and continue the loop with the extended sequence.
        input_ids = torch.cat([input_ids, next_token], dim=-1)

    return tokenizer.decode(input_ids[0], skip_special_tokens=True)


print(generate_text("Deep learning is fundamentally about"))

Deep learning is fundamentally about finding and finding a solution to a problem. If you are already using deep learning and it is not being used for specific tasks, then you should not


## 8. Conclusion

This notebook reimplemented the core decoding mechanics that sit underneath
high-level APIs like `model.generate()`:

- Raw logits were converted into shaped probability distributions via
  **temperature scaling**.
- The candidate pool was narrowed via **Top-K** (fixed cutoff) and
  **Top-P** (adaptive, confidence-aware cutoff) masking.
- These primitives were composed into a working **autoregressive sampling
  loop** on top of GPT-2.

**Possible extensions:**
- Add repetition penalties or no-repeat n-gram blocking.
- Benchmark perplexity/diversity trade-offs across temperature and top-p settings.
- Extend the loop to batched generation for multiple prompts at once.